In [1]:
pip install requests beautifulsoup4 lxml

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import uuid
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36"
}

SAVE_DIR = "sidewalk_images"
os.makedirs(SAVE_DIR, exist_ok=True)

def download_image(url, filename):
    try:
        r = requests.get(url, headers=HEADERS, timeout=10)
        if r.status_code == 200:
            with open(filename, "wb") as f:
                f.write(r.content)
            print("  => Đã lưu:", filename)
    except:
        pass

def crawl_article(article_url):
    try:
        response = requests.get(article_url, headers=HEADERS)
        soup = BeautifulSoup(response.text, "lxml")

        article_body = soup.find("article", class_="fck_detail")
        if not article_body:
            article_body = soup

        imgs = article_body.find_all("img")

        # BỘ TỪ KHÓA NHẬN DIỆN ẢNH VI PHẠM (Dành cho Object Detection / VQA)
        violation_keywords = [
            # Hành vi
            "lấn chiếm", "vi phạm", "chiếm dụng", "bày bán", "kinh doanh", "bát nháo", 
            "nhếch nhác", "bịt kín", "tràn xuống lòng đường", "chướng ngại vật",
            
            # Đối tượng vi phạm (Bounding Box / Object)
            "bàn ghế", "hàng rong", "xe đẩy", "mái che", "mái vẩy", "biển quảng cáo", 
            "xe máy", "ô tô", "đỗ xe", "đậu xe", "bãi gửi xe", "giữ xe", "vật liệu xây dựng",
            
            # Động thái của cơ quan chức năng (Classification/Context)
            "xử phạt", "dẹp loạn", "tịch thu", "cẩu xe", "giành lại", "lập biên bản",
            
            # Hậu quả
            "không còn lối đi", "người đi bộ", "trật tự đô thị"
        ]

        for img in imgs:
            src = img.get("data-src") or img.get("src")
            if not src: continue
            
            src = urljoin(article_url, src)
            if any(x in src.lower() for x in ["logo", "icon", "avatar", "ads", ".svg", "gif"]):
                continue

            alt_text = img.get("alt", "").lower()
            parent = img.parent
            while parent and parent.name not in ['figure', 'table', 'div']:
                parent = parent.parent
            caption_text = parent.text.lower() if parent else ""

            context_text = alt_text + " " + caption_text

            # Lọc: Có chứa ít nhất 1 từ khóa vi phạm
            if not any(kw in context_text for kw in violation_keywords):
                continue

            clean_path = urlparse(src).path
            ext = clean_path.split(".")[-1]
            if len(ext) > 4 or not ext.isalnum(): ext = "jpg"

            filename = os.path.join(SAVE_DIR, f"{uuid.uuid4().hex[:8]}.{ext}")
            download_image(src, filename)

    except Exception as e:
        pass

def crawl_violation_images(max_pages=3):
    # BỘ TOPIC TÌM KIẾM PHONG PHÚ ĐỂ VÉT CẠN DỮ LIỆU
    topics = [
        # Các từ khóa phổ biến nhất
        "lấn chiếm vỉa hè", "dẹp vỉa hè", "giành lại vỉa hè", "chiếm dụng vỉa hè",
        
        # Nhóm kinh doanh, buôn bán
        "bán hàng rong vỉa hè", "kinh doanh vỉa hè", "buôn bán lề đường", 
        "chợ tự phát vỉa hè", "quán nhậu lấn vỉa hè", "bày biện trên vỉa hè",
        
        # Nhóm phương tiện, dừng đỗ
        "đỗ xe trên vỉa hè", "đậu xe lề đường", "bãi giữ xe vỉa hè", "thu phí vỉa hè",
        "cẩu xe đỗ sai quy định", "vi phạm giao thông tĩnh",
        
        # Nhóm xử lý, chiến dịch
        "xử phạt vỉa hè", "lấn chiếm lòng lề đường", "trật tự đô thị vỉa hè", 
        "dọn dẹp vỉa hè", "đòi lại vỉa hè", "trả lại vỉa hè",
        
        # Nhóm miêu tả hiện trạng (từ báo chí hay dùng)
        "bát nháo vỉa hè", "bức tử vỉa hè", "vỉa hè bị băm nát", 
        "người đi bộ xuống lòng đường", "bậc tam cấp lấn vỉa hè"
    ]

    
    article_urls = []
    
    for topic in topics:
        print(f"[*] Đang tìm kiếm cụm từ: '{topic}'")
        for page in range(1, max_pages + 1):
            search_url = "https://timkiem.vnexpress.net/"
            params = {"q": topic, "page": page}
            try:
                r = requests.get(search_url, params=params, headers=HEADERS)
                soup = BeautifulSoup(r.text, "lxml")
                for h3 in soup.find_all("h3", class_="title-news"):
                    a_tag = h3.find("a")
                    if a_tag and "vnexpress.net" in a_tag.get("href", ""):
                        article_urls.append(a_tag.get("href"))
            except:
                pass

    article_urls = list(set(article_urls))
    print(f"\n[+] Đã gộp và loại bỏ link trùng. Tổng cộng {len(article_urls)} bài viết.")
    print("[-] Bắt đầu chắt lọc ảnh vi phạm...\n")

    for i, url in enumerate(article_urls, 1):
        print(f"[{i}/{len(article_urls)}] {url}")
        crawl_article(url)
        
    print("\n[OK] Hoàn tất thu thập Dataset Vi Phạm Vỉa Hè!")

# Cài đặt số lượng trang tìm kiếm mỗi Topic (khuyến nghị 3-5 trang)
crawl_violation_images(max_pages=100)


[*] Đang tìm kiếm cụm từ: 'lấn chiếm vỉa hè'
[*] Đang tìm kiếm cụm từ: 'dẹp vỉa hè'
[*] Đang tìm kiếm cụm từ: 'giành lại vỉa hè'
[*] Đang tìm kiếm cụm từ: 'chiếm dụng vỉa hè'
[*] Đang tìm kiếm cụm từ: 'bán hàng rong vỉa hè'
[*] Đang tìm kiếm cụm từ: 'kinh doanh vỉa hè'
[*] Đang tìm kiếm cụm từ: 'buôn bán lề đường'
[*] Đang tìm kiếm cụm từ: 'chợ tự phát vỉa hè'
[*] Đang tìm kiếm cụm từ: 'quán nhậu lấn vỉa hè'
[*] Đang tìm kiếm cụm từ: 'bày biện trên vỉa hè'
[*] Đang tìm kiếm cụm từ: 'đỗ xe trên vỉa hè'
[*] Đang tìm kiếm cụm từ: 'đậu xe lề đường'
[*] Đang tìm kiếm cụm từ: 'bãi giữ xe vỉa hè'
[*] Đang tìm kiếm cụm từ: 'thu phí vỉa hè'
[*] Đang tìm kiếm cụm từ: 'cẩu xe đỗ sai quy định'
[*] Đang tìm kiếm cụm từ: 'vi phạm giao thông tĩnh'
[*] Đang tìm kiếm cụm từ: 'xử phạt vỉa hè'
[*] Đang tìm kiếm cụm từ: 'lấn chiếm lòng lề đường'
[*] Đang tìm kiếm cụm từ: 'trật tự đô thị vỉa hè'
[*] Đang tìm kiếm cụm từ: 'dọn dẹp vỉa hè'
[*] Đang tìm kiếm cụm từ: 'đòi lại vỉa hè'
[*] Đang tìm kiếm cụm từ: 